# RSO-112: Analyse thermal and stability indicators during Dome louver testing

During the shutdown period (31.1. -14.2) we run multiple versions (different louver configurations) of the BLOCK-T679. Even though we were not on sky and we don’t have the IQ data, we would like to see the inside dome temperature response for different louvers configurations. 

Analyze dome and telescope thermal response during the 6-louver experimental campaigns, focusing on temperature coupling and ventilation behavior in the absence of image-quality (FWHM) indicators.

**Descripcion**

Create a small set of easy-to-interpret temperature indicators that describe how the dome and telescope behave thermally during each louvers configuration.

 

**Expected results:**

New variables:

inside vs outside temperature difference

telescope vs outside temperature difference

temperature gradient across the telescope sensors (111-113)

Table: basic statistics per configuration (average, variation)

Plots: simple box or line plots showing how these indicators change between configurations

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from astropy.time import Time
import matplotlib.dates as mdates
import matplotlib.cm as cm
import seaborn as sns

from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState

In [ ]:
t_start_period = Time("2026-01-31T00:00:00Z", scale="utc")
t_end_period = Time("2026-02-15T00:00:00Z", scale="utc")

efd_client = makeEfdClient()

In [ ]:
# make a list of all topics in the EFD related to MTMount
topics = await efd_client.get_topics()
for topic in topics:
    if 'ESS' in topic:
        print(topic)

# Queries

In [ ]:
def query_setlouvers(start, end):
    df_louvers = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.command_setLouvers",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_louvers

In [ ]:
def query_essTemperature(start, end):
    df_esstemperature = getEfdData(
        client=efd_client,
        topic="lsst.sal.ESS.temperature",
        columns=['location', 'private_identity','sensorName', 'temperatureItem0','timestamp'],
        begin=start,
        end=end,
    )

    return df_esstemperature

# Plot functions

In [ ]:
def plot_temperature_with_louvers(df_temp, df_setlouvers):

    # Number of unique configuration
    conf_ids = sorted(df_setlouvers['louvers_conf'].unique())
    n_confs = len(conf_ids)

    # Colormap by configuration
    colors = plt.get_cmap('tab20', n_confs)

    # Columns positions
    position_cols = df_setlouvers.filter(regex=r'^position').columns
    position_cols = sorted(position_cols, key=lambda x: int(x.replace('position','')))

    # Labels for configuration
    def build_conf_label(row):
        config = row[position_cols]
        non_zero = config[config != 0]
        if len(non_zero) == 0:
            return f"Configuration {row['louvers_conf']}: all closed"
        lines = [f"Configuration {row['louvers_conf']}:"]
        for col, val in non_zero.items():
            lines.append(f"{col}: {int(val)}")
        return "\n".join(lines)

    df_setlouvers = df_setlouvers.copy()
    df_setlouvers['conf_label'] = df_setlouvers.apply(build_conf_label, axis=1)

    # Dictionary to rename sensors
    sensor_labels = {
        "ESS:111": "Temperature dome inside",
        "ESS:301": "Temperature ess weather station",
        "ESS:113": "Temperature m1m3 inside",
        "ESS:112": "Temperature m2 inside"
    }

    for conf_num in conf_ids:

        fig, ax = plt.subplots(figsize=(10,6))

        # Plot temperature sensors
        for sensor_id, group in df_temp.groupby('private_identity'):
            label = sensor_labels.get(sensor_id, sensor_id)
            ax.plot(group['time'], group['temperatureItem0'], label=label)

        # Plot configuration intervals
        conf_df = df_setlouvers[df_setlouvers['louvers_conf'] == conf_num]

        for _, row in conf_df.iterrows():
            start = row['time_stamp']
            end = row['time_end']
            label = row['conf_label']
            ax.axvspan(start, end, color=colors(int(conf_num)-1), alpha=0.3, label=label)

        ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m'))

        plt.xlabel("Day")
        plt.ylabel("Temperature (°C)")
        plt.title(f"Temperature - Louver Configuration {conf_num}")

        # Remove duplicated legend entries
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.02,1), loc='upper left')

        ax.grid(True)
        plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()

# Configuration of louvers

In [ ]:
df_setlouvers = query_setlouvers(t_start_period, t_end_period)

In [ ]:
df_setlouvers.head()

In [ ]:
# Copy current index into a new column before any merge
df_setlouvers['time_stamp'] = df_setlouvers.index

In [ ]:
# Select all columns that start with "position"
position_cols = df_setlouvers.filter(regex=r'^position').columns

# Sort columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position', '')))

# Compute unique combinations
combination_counts = (
    df_setlouvers[position_cols]
    .value_counts()
    .reset_index(name='count')
)

# Create configuration ID (1 to N)
combination_counts['louvers_conf'] = range(1, len(combination_counts) + 1)

# Merge configuration ID back into original dataframe
df_setlouvers = df_setlouvers.merge(
    combination_counts[position_cols + ['louvers_conf']],
    on=position_cols,
    how='left'
)

print(f"Number of unique configurations detected: {len(combination_counts)}\n")

# Print configurations showing only non-zero positions
for _, row in combination_counts.iterrows():
    
    conf_id = row['louvers_conf']
    count = row['count']
    
    print(f"Configuration {conf_id} (appears {count} times):")
    
    # Extract position values
    config = row[position_cols]
    
    # Keep only non-zero values
    non_zero = config[config != 0]
    
    if len(non_zero) == 0:
        print("  All positions are 0")
    else:
        for col, val in non_zero.items():
            print(f"  {col}: {val}")
    
    print("-" * 40)

15 combinations have been made, varying the opening of the louvers: 2, 11, 12, 20, 21, and 29.

In [ ]:
df_setlouvers.head()

# ESS Temperature

In [ ]:
df_esstemperature = query_essTemperature(t_start_period, t_end_period)

In [ ]:
df_esstemperature.head()

## Inside vs outside temperature difference

ess_weather_station_sal_index = 301
m1m3_inside_cell_sal_index = 113
dome_inside_sal_index = 111


In [ ]:
# Keep only the ESS identities with 111, 112 and 113
df_temp_inandout = df_esstemperature[
    df_esstemperature['private_identity'].isin(['ESS:111','ESS:301'])
].copy()

In [ ]:
df_temp_inandout.head()

### Plots of telescope sensors (111 and 301)

In [ ]:
# Make a copy
df_temp_inandout = df_temp_inandout.copy()

# Convert timestamp to datetime
df_temp_inandout['time'] = pd.to_datetime(df_temp_inandout['timestamp'], unit='s')

# Sort by time
df_temp_inandout = df_temp_inandout.sort_values('time')

# Dictionary to rename sensors in the legend
sensor_labels = {
    "ESS:111": "Temperature Inside",
    "ESS:301": "Temperature Outside"
}

fig, ax = plt.subplots(figsize=(10,6))

for sensor_id, group in df_temp_inandout.groupby('private_identity'):
    
    label = sensor_labels.get(sensor_id, sensor_id)  # fallback to original name
    
    ax.plot(group['time'], group['temperatureItem0'], label=label)

# Force one tick per day
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))

# Format showing day and month
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m'))

plt.xlabel("Day")
plt.ylabel("Temperature (°C)")
plt.title("Temperature Inside and outside")

plt.legend() 
plt.grid(True)

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### Cross with the louver settings

In [ ]:
# Make copies to avoid modifying original slices
df_temp_inandout = df_temp_inandout.copy()
df_setlouvers = df_setlouvers.copy()

# Convert timestamps
df_temp_inandout['time'] = pd.to_datetime(df_temp_inandout['timestamp'], unit='s')
df_setlouvers['time_stamp'] = pd.to_datetime(df_setlouvers['time_stamp'])

# Sort data
df_temp_inandout = df_temp_inandout.sort_values('time')
df_setlouvers = df_setlouvers.sort_values('time_stamp')

# Dictionary to rename sensors in the legend
sensor_labels = {
    "ESS:111": "Temperature Inside",
    "ESS:301": "Temperature Outside"
}

fig, ax = plt.subplots(figsize=(10,6))

for sensor_id, group in df_temp_inandout.groupby('private_identity'):
    
    label = sensor_labels.get(sensor_id, sensor_id)  # fallback to original name
    
    ax.plot(group['time'], group['temperatureItem0'], label=label)

# Add vertical lines for louver configuration changes
for t in df_setlouvers['time_stamp']:
    ax.axvline(t, linestyle='--', linewidth=1, alpha=0.5)

# One tick per day
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m'))

plt.xlabel("Day")
plt.ylabel("Temperature (°C)")
plt.title("Temperature Inside and outside with Louver Configuration")
plt.grid(True)
plt.legend()

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
df_setlouvers['time_end'] = df_setlouvers['time_stamp'].shift(-1)
max_time = df_temp_inandout['time'].max()
df_setlouvers['time_end'] = df_setlouvers['time_end'].fillna(max_time)

In [ ]:
# Column positions
position_cols = df_setlouvers.filter(regex=r'^position').columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position','')))

# Function to labels
def build_conf_label(row):
    config = row[position_cols]
    non_zero = config[config != 0]

    if len(non_zero) == 0:
        return f"Configuration {row['louvers_conf']}: all closed"

    lines = [f"Configuration {row['louvers_conf']}:"]
    for col, val in non_zero.items():
        lines.append(f"{col}: {int(val)}")
    return "\n".join(lines)

df_setlouvers['conf_label'] = df_setlouvers.apply(build_conf_label, axis=1)

In [ ]:
# Colormap for configuration
n_confs = df_setlouvers['louvers_conf'].nunique()
colors = plt.colormaps['tab20'].resampled(n_confs)

df_temp_inandout['time'] = pd.to_datetime(df_temp_inandout['time'], utc=True)
df_setlouvers['time_stamp'] = pd.to_datetime(df_setlouvers['time_stamp'], utc=True)
df_setlouvers['time_end'] = pd.to_datetime(df_setlouvers['time_end'], utc=True)

df_temp_inandout['time'] = df_temp_inandout['time'].dt.tz_convert(None)
df_setlouvers['time_stamp'] = df_setlouvers['time_stamp'].dt.tz_convert(None)
df_setlouvers['time_end'] = df_setlouvers['time_end'].dt.tz_convert(None)


# Dictionary to rename sensors in the legend
sensor_labels = {
    "ESS:111": "Temperature Inside",
    "ESS:301": "Temperature Outside"
}

fig, ax = plt.subplots(figsize=(10,6))

for sensor_id, group in df_temp_inandout.groupby('private_identity'):
    
    label = sensor_labels.get(sensor_id, sensor_id)  # fallback to original name
    
    ax.plot(group['time'], group['temperatureItem0'], label=label)

# --- Shading by configuration ---
for i, (_, row) in enumerate(df_setlouvers.iterrows()):
    start = row['time_stamp']
    end = row['time_end']
    conf_num = int(row['louvers_conf']) - 1 
    ax.axvspan(start, end, color=colors(conf_num), alpha=0.15)

ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m'))

plt.xlabel("Day")
plt.ylabel("Temperature (°C)")
plt.title("Temperature Inside and outside with Louver Configuration (colored)")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plot_temperature_with_louvers(df_temp_inandout, df_setlouvers)

Table: basic statistics per configuration (average, variation)

In [ ]:
# Format of the times
df_temp_inandout['time'] = pd.to_datetime(df_temp_inandout['time'])
df_setlouvers['time_stamp'] = pd.to_datetime(df_setlouvers['time_stamp'])
df_setlouvers['time_end'] = pd.to_datetime(df_setlouvers['time_end'])

In [ ]:
results = []

for _, conf in df_setlouvers.iterrows():

    start = conf['time_stamp']
    end = conf['time_end']
    conf_id = conf['louvers_conf']
    conf_label = conf['conf_label']

    # datos dentro del intervalo
    subset = df_temp_inandout[(df_temp_inandout['time'] >= start) & (df_temp_inandout['time'] <= end)]

    duration = (end - start).total_seconds()/60  # minutos

    for sensor, group in subset.groupby('private_identity'):

        temp = group['temperatureItem0']

        if len(temp) < 2:
            continue

        gradient = (temp.iloc[-1] - temp.iloc[0]) / duration

        results.append({
            "louvers_conf": conf_id,
            "conf_label": conf_label,
            "sensor": sensor,
            "duration_min": duration,
            "mean_temp": temp.mean(),
            "std_temp": temp.std(),
            "min_temp": temp.min(),
            "max_temp": temp.max(),
            "temp_range": temp.max() - temp.min(),
            "gradient_deg_per_min": gradient
        })

df_stats = pd.DataFrame(results)

In [ ]:
df_stats

# ESS Temperature:  telescope sensors (111-113)

In [ ]:
# Keep only the ESS identities with 111, 112 and 113
df_ess = df_esstemperature[
    df_esstemperature['private_identity'].isin(['ESS:111','ESS:112','ESS:113'])
].copy()

In [ ]:
df_ess.head()

### Plots of telescope sensors (111-113)

In [ ]:
# Make a copy
df_ess = df_ess.copy()

# Convert timestamp to datetime
df_ess['time'] = pd.to_datetime(df_ess['timestamp'], unit='s')

# Sort by time
df_ess = df_ess.sort_values('time')

fig, ax = plt.subplots(figsize=(10,6))

for sensor_id, group in df_ess.groupby('private_identity'):
    ax.plot(group['time'], group['temperatureItem0'], label=sensor_id)

# Force one tick per day
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))

# Format showing day and month
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m'))

plt.xlabel("Day")
plt.ylabel("Temperature (°C)")
plt.title("ESS Temperature Sensors")
plt.legend(title="Sensor")
plt.grid(True)

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### Cross with the louver settings

In [ ]:
# Make copies to avoid modifying original slices
df_ess = df_ess.copy()
df_setlouvers = df_setlouvers.copy()

# Convert timestamps
df_ess['time'] = pd.to_datetime(df_ess['timestamp'], unit='s')
df_setlouvers['time_stamp'] = pd.to_datetime(df_setlouvers['time_stamp'])

# Sort data
df_ess = df_ess.sort_values('time')
df_setlouvers = df_setlouvers.sort_values('time_stamp')

fig, ax = plt.subplots(figsize=(10,6))

# Plot temperature lines
for sensor_id, group in df_ess.groupby('private_identity'):
    ax.plot(group['time'], group['temperatureItem0'], label=sensor_id)

# Add vertical lines for louver configuration changes
for t in df_setlouvers['time_stamp']:
    ax.axvline(t, linestyle='--', linewidth=1, alpha=0.5)

# One tick per day
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m'))

plt.xlabel("Day")
plt.ylabel("Temperature (°C)")
plt.title("ESS Temperature Sensors with Louver Configuration")
plt.legend(title="Sensor")
plt.grid(True)

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
df_setlouvers['time_end'] = df_setlouvers['time_stamp'].shift(-1)
max_time = df_ess['time'].max()
df_setlouvers['time_end'] = df_setlouvers['time_end'].fillna(max_time)

In [ ]:
# ESTO SE PODRÏA BORRAR
# Column positions
position_cols = df_setlouvers.filter(regex=r'^position').columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position','')))

# Function to labels
def build_conf_label(row):
    config = row[position_cols]
    non_zero = config[config != 0]

    if len(non_zero) == 0:
        return f"Configuration {row['louvers_conf']}: all closed"

    lines = [f"Configuration {row['louvers_conf']}:"]
    for col, val in non_zero.items():
        lines.append(f"{col}: {int(val)}")
    return "\n".join(lines)

df_setlouvers['conf_label'] = df_setlouvers.apply(build_conf_label, axis=1)

In [ ]:
# Colormap for configuration
n_confs = df_setlouvers['louvers_conf'].nunique()
colors = plt.colormaps['tab20'].resampled(n_confs)

df_ess['time'] = pd.to_datetime(df_ess['time'], utc=True)
df_setlouvers['time_stamp'] = pd.to_datetime(df_setlouvers['time_stamp'], utc=True)
df_setlouvers['time_end'] = pd.to_datetime(df_setlouvers['time_end'], utc=True)

df_ess['time'] = df_ess['time'].dt.tz_convert(None)
df_setlouvers['time_stamp'] = df_setlouvers['time_stamp'].dt.tz_convert(None)
df_setlouvers['time_end'] = df_setlouvers['time_end'].dt.tz_convert(None)




fig, ax = plt.subplots(figsize=(12,6))

# --- Temperature Lines ---
for sensor_id, group in df_ess.groupby('private_identity'):
    ax.plot(group['time'], group['temperatureItem0'], label=sensor_id)

# --- Shading by configuration ---
for i, (_, row) in enumerate(df_setlouvers.iterrows()):
    start = row['time_stamp']
    end = row['time_end']
    conf_num = int(row['louvers_conf']) - 1 
    ax.axvspan(start, end, color=colors(conf_num), alpha=0.15)

ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m'))

plt.xlabel("Day")
plt.ylabel("Temperature (°C)")
plt.title("ESS Temperature Sensors with Louver Configurations (colored)")
plt.legend(title="Sensor")
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Number of unique configuration
conf_ids = sorted(df_setlouvers['louvers_conf'].unique())
n_confs = len(conf_ids)

# Colormap by configuration
colors = plt.get_cmap('tab20', n_confs)

# Columns positions
position_cols = df_setlouvers.filter(regex=r'^position').columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position','')))

# Labels for configuration
def build_conf_label(row):
    config = row[position_cols]
    non_zero = config[config != 0]
    if len(non_zero) == 0:
        return f"Configuration {row['louvers_conf']}: all closed"
    lines = [f"Configuration {row['louvers_conf']}:"]
    for col, val in non_zero.items():
        lines.append(f"{col}: {int(val)}")
    return "\n".join(lines)

df_setlouvers['conf_label'] = df_setlouvers.apply(build_conf_label, axis=1)

for conf_num in conf_ids:

    fig, ax = plt.subplots(figsize=(12,6))
    
    for sensor_id, group in df_ess.groupby('private_identity'):
        ax.plot(group['time'], group['temperatureItem0'], label=f"Sensor {sensor_id}")
    
    # A plot for configuration
    conf_df = df_setlouvers[df_setlouvers['louvers_conf'] == conf_num]

    for i, (_, row) in enumerate(conf_df.iterrows()):
        start = row['time_stamp']
        end = row['time_end']
        label = row['conf_label']
        ax.axvspan(start, end, color=colors(int(conf_num)-1), alpha=0.3, label=label)
    
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m'))

    plt.xlabel("Day")
    plt.ylabel("Temperature (°C)")
    plt.title(f"ESS Temperatures - Louver Configuration {conf_num}")
    
    # Avoid duplicate labels
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.02,1), loc='upper left')
    
    ax.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

Table: basic statistics per configuration (average, variation)

In [ ]:
# Format of the times
df_ess['time'] = pd.to_datetime(df_ess['time'])
#df_setlouvers['time_stamp'] = pd.to_datetime(df_setlouvers['time_stamp'])
#df_setlouvers['time_end'] = pd.to_datetime(df_setlouvers['time_end'])

In [ ]:
results = []

for _, conf in df_setlouvers.iterrows():

    start = conf['time_stamp']
    end = conf['time_end']
    conf_id = conf['louvers_conf']
    conf_label = conf['conf_label']

    # datos dentro del intervalo
    subset = df_ess[(df_ess['time'] >= start) & (df_ess['time'] <= end)]

    duration = (end - start).total_seconds()/60  # minutos

    for sensor, group in subset.groupby('private_identity'):

        temp = group['temperatureItem0']

        if len(temp) < 2:
            continue

        gradient = (temp.iloc[-1] - temp.iloc[0]) / duration

        results.append({
            "louvers_conf": conf_id,
            "conf_label": conf_label,
            "sensor": sensor,
            "duration_min": duration,
            "mean_temp": temp.mean(),
            "std_temp": temp.std(),
            "min_temp": temp.min(),
            "max_temp": temp.max(),
            "temp_range": temp.max() - temp.min(),
            "gradient_deg_per_min": gradient
        })

df_stats = pd.DataFrame(results)

In [ ]:
df_stats

In [ ]:
# Separate in tables for louvers configuration
tables = {}

for conf in df_stats['louvers_conf'].unique():
    tables[conf] = df_stats[df_stats['louvers_conf']==conf]

In [ ]:
# For example, to see configuration 1 for the 3 sensors and the stat each time this configuration is working
tables[1]

In [ ]:
# Summary of the different configuration of louvers
df_summary = (
    df_stats
    .groupby(['louvers_conf','conf_label','sensor'])
    .agg({
        'duration_min':'mean',
        'mean_temp':'mean',
        'std_temp':'mean',
        'gradient_deg_per_min':'mean',
        'temp_range':'mean'
    })
    .reset_index()
)

In [ ]:
df_summary

In [ ]:
df_stats.columns

In [ ]:
df_stats.head()

In [ ]:
# Mean Temperature per Louver Configuration
# Shows which configuration leads to higher or lower temperatures

plt.figure(figsize=(12,6))

sns.barplot(
    data=df_stats,
    x="louvers_conf",
    y="mean_temp",
    hue="sensor"
)

plt.xlabel("Louver Configuration")
plt.ylabel("Mean Temperature (°C)")
plt.title("Mean Temperature per Louver Configuration")

plt.xticks(rotation=45)
plt.grid(True)

# ---- Create configuration legend text ----
conf_map = (
    df_setlouvers[['louvers_conf','conf_label']]
    .drop_duplicates()
    .sort_values('louvers_conf')
)

text = "\n".join(
    f"{row.louvers_conf} → {row.conf_label}"
    for _, row in conf_map.iterrows()
)

# ---- Add configuration explanation outside the plot ----
plt.gcf().text(
    1.02,
    0.5,
    "Configuration meaning:\n\n" + text,
    fontsize=9,
    va="center"
)

plt.tight_layout()
plt.show()

In [ ]:
# Temperature Stability per Louver Configuration
# Lower standard deviation means more stable thermal behavior

plt.figure(figsize=(12,6))

sns.barplot(
    data=df_stats,
    x="louvers_conf",
    y="std_temp",
    hue="sensor"
)

plt.xlabel("Louver Configuration")
plt.ylabel("Temperature Standard Deviation (°C)")
plt.title("Temperature Stability per Louver Configuration")
plt.xticks(rotation=45)
plt.grid(True)

# ---- Create configuration legend text ----
conf_map = (
    df_setlouvers[['louvers_conf','conf_label']]
    .drop_duplicates()
    .sort_values('louvers_conf')
)

text = "\n".join(
    f"{row.louvers_conf} → {row.conf_label}"
    for _, row in conf_map.iterrows()
)

# ---- Add configuration explanation outside the plot ----
plt.gcf().text(
    1.02,
    0.5,
    "Configuration meaning:\n\n" + text,
    fontsize=9,
    va="center"
)


plt.tight_layout()
plt.show()

In [ ]:
# Temperature Gradient per Louver Configuration
# Positive values indicate temperature increase, negative values indicate cooling

plt.figure(figsize=(12,6))

sns.barplot(
    data=df_stats,
    x="louvers_conf",
    y="gradient_deg_per_min",
    hue="sensor"
)

plt.xlabel("Louver Configuration")
plt.ylabel("Temperature Gradient (°C/min)")
plt.title("Temperature Gradient per Louver Configuration")
plt.xticks(rotation=45)
plt.axhline(0, color='black', linestyle='--')


# ---- Create configuration legend text ----
conf_map = (
    df_setlouvers[['louvers_conf','conf_label']]
    .drop_duplicates()
    .sort_values('louvers_conf')
)

text = "\n".join(
    f"{row.louvers_conf} → {row.conf_label}"
    for _, row in conf_map.iterrows()
)

# ---- Add configuration explanation outside the plot ----
plt.gcf().text(
    1.02,
    0.5,
    "Configuration meaning:\n\n" + text,
    fontsize=9,
    va="center"
)

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature Distribution per Louver Configuration
# Shows variability and outliers across repeated occurrences

plt.figure(figsize=(12,6))

sns.boxplot(
    data=df_stats,
    x="louvers_conf",
    y="mean_temp",
    hue="sensor"
)

plt.xlabel("Louver Configuration")
plt.ylabel("Mean Temperature (°C)")
plt.title("Temperature Distribution per Louver Configuration")
plt.xticks(rotation=45)

# ---- Create configuration legend text ----
conf_map = (
    df_setlouvers[['louvers_conf','conf_label']]
    .drop_duplicates()
    .sort_values('louvers_conf')
)

text = "\n".join(
    f"{row.louvers_conf} → {row.conf_label}"
    for _, row in conf_map.iterrows()
)

# ---- Add configuration explanation outside the plot ----
plt.gcf().text(
    1.02,
    0.5,
    "Configuration meaning:\n\n" + text,
    fontsize=9,
    va="center"
)


plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature Stability Distribution per Configuration
# Compares temperature variability across configurations

plt.figure(figsize=(12,6))

sns.boxplot(
    data=df_stats,
    x="louvers_conf",
    y="std_temp",
    hue="sensor"
)

plt.xlabel("Louver Configuration")
plt.ylabel("Temperature Standard Deviation (°C)")
plt.title("Temperature Stability Distribution per Configuration")
plt.xticks(rotation=45)

# ---- Create configuration legend text ----
conf_map = (
    df_setlouvers[['louvers_conf','conf_label']]
    .drop_duplicates()
    .sort_values('louvers_conf')
)

text = "\n".join(
    f"{row.louvers_conf} → {row.conf_label}"
    for _, row in conf_map.iterrows()
)

# ---- Add configuration explanation outside the plot ----
plt.gcf().text(
    1.02,
    0.5,
    "Configuration meaning:\n\n" + text,
    fontsize=9,
    va="center"
)

plt.grid(True)
plt.tight_layout()
plt.show()

####*********** SUCIO*****

In [ ]:
def query_thermal(start, end):
    df_thermal = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.thermal",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_thermal

In [ ]:
df_thermal = query_thermal(t_start_period, t_end_period)

In [ ]:
df_thermal.head()

In [ ]:
df.columns

In [ ]:
# Make sure time columns are datetime (if not already)
df_setlouvers['time_stamp'] = pd.to_datetime(df_setlouvers['time_stamp'])
df_thermal['timestamp'] = pd.to_datetime(df_thermal['timestamp'])

# Sort both dataframes by time
df_setlouvers = df_setlouvers.sort_values('time_stamp')
df_thermal = df_thermal.sort_values('timestamp')

In [ ]:
# Perform temporal merge
df_thermal_with_conf = pd.merge_asof(
    df_thermal,
    df_setlouvers[['time_stamp', 'louvers_conf']],
    left_on='timestamp',
    right_on='time_stamp',
    direction='backward'  # take last known configuration
)

In [ ]:
def query_weather(start, end):
    df_weather = getEfdData(
        client=efd_client,
        topic="lsst.sal.WeatherForecast.dailyTrend",
        columns=["Max temperature", "temperatureMin", "Mean temperature", ],
        begin=start,
        end=end,
    )

    return df_weather

In [ ]:
df = query_weather(t_start_period, t_end_period)

## *****************************

In [ ]:
# Esto debe tener demasiado datos y no carga ni cambiando las fechas
def logevent_louversEnabled(start, end):
    df_louvers = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.louvers",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_louvers

In [ ]:
df = logevent_louversEnabled(t_start_period, t_end_period)

In [ ]:
df

In [ ]:
## Insert here the dayObs of interest
dayObs = 20260208# 20231122

In [ ]:
# Select data from a given date
eventMaker = TMAEventMaker()
events = eventMaker.getEvents(dayObs)


# Get events related to soak tests (block 137 currently)
blockT679 = []
for event in events:
    blockInfos = event.blockInfos
    if blockInfos is None:
        continue  # no block info attached to event at all

    # check if any of the attached blockInfos are for block 137
    blockNums = {b.blockNumber for b in blockInfos}
    print(blockNums)
    if T679 in blockNums:
        blockT679.append(event)

print(f"Of the {len(events)} events, {len(blockT679)} relate to block BLOCK-T679.")

In [ ]:
from lsst.summit.utils.blockUtils import BlockParser
from lsst.summit.utils.tmaUtils import TMAEventMaker

In [ ]:
# Set the day_obs list 
day_obs_list = range(20250127, 20260215 + 1)

# For the TMA events 
event_maker = TMAEventMaker()

In [ ]:
# For each day_obs in the list determine which blocks were run and put
# the list of blocks into the block_list.
 
block_list = []

for day_obs in day_obs_list:
    block_parser = BlockParser(day_obs)
    blocks = block_parser.getBlockNums()
    block_list.append(blocks)

# Put the variable length nested list into an awkward array and then 
# put that into a pandas dataframe with the awkward array extension
# so that the list of blocks is shown in a column.
blocks = ak.Array({"day_obs": day_obs_list, "blocks": block_list})
series = akpd.from_awkward(blocks)
pandas_df = series.ak.to_columns(extract_all=True)
pandas_df

In [ ]:
def day_obs_report(day_obs):
    '''
    Loop over the blocks and sequences for one day and produce a report.
    Interspace TMA events with the block info.
    '''

    block_parser = BlockParser(day_obs)
    tma_events = event_maker.getEvents(day_obs)
    blocks = block_parser.getBlockNums()

    print(f'SUMMARY REPORT FOR DAYOBS: {day_obs} \n')
    print(blocks)
    for block_id in blocks:
        sequences =  block_parser.getSeqNums(block_id)

        print(f'BLOCK:SEQ \t STATES')

        for seq_id in sequences:
            info = block_parser.getBlockInfo(block_id, seq_id)
            state_string = ' '.join([str(state) for state in info.states])
            print(f'{info.blockNumber}:{info.seqNum} \t\t {state_string}')

            # Also print any TMA events for this block/sequence
            event = block_parser.getEventsForBlock(tma_events, block_id, seq_id)
            if event: print(event)

        print(f'\n')

In [ ]:
day_obs_list = range(20260201, 20260214 + 1)

for day_obs in day_obs_list:
    day_obs_report(day_obs)